# MSRE — Event Study Research Notebook

**Purpose:** Interactive exploration of the Macro Shock Risk Engine pipeline.
Walk through a complete risk assessment for a historical event and inspect all intermediate outputs.

**Environment:** Research only. No live feeds. Uses synthetic market data.

**Note:** This notebook is for research and exploration. Production code lives in `python/src/`.

In [ ]:
import sys
sys.path.insert(0, '../python/src')
sys.path.insert(0, '../examples')

from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from macro_shock.orchestration.pipeline import MacroShockPipeline
from macro_shock.data.ingestion import MarketStateBuilder
from synthetic_data_generator import generate_synthetic_events, generate_market_data_df

import warnings
warnings.filterwarnings('ignore')

print('Environment ready.')

## 1. Initialize Pipeline

In [ ]:
config = {
    'use_transformer': False,
    'audit_log_dir': '/tmp/msre_notebook',
    'fail_fast': False,
}
pipeline = MacroShockPipeline(config=config, environment='research')
print('Pipeline initialized.')

## 2. Define the Event

We'll analyze a synthetic weekend emergency event — the most challenging scenario for portfolio risk management.

In [ ]:
event_raw = {
    'title': 'Emergency Federal Reserve Statement — Weekend',
    'institution': 'Federal Reserve',
    'speaker': 'Jerome Powell',
    'speaker_role': 'Chair',
    'event_time': '2024-03-09T19:00:00+00:00',  # Saturday evening
    'description': 'Emergency weekend statement on banking sector stress.',
    'headline_summary': 'Fed announces emergency backstop citing systemic risk and financial stability concerns.',
    'prepared_remarks': (
        'The Federal Reserve Board has voted unanimously to establish emergency lending facilities '
        'to address financial stability concerns in the banking system. '
        'We are taking extraordinary measures to ensure liquidity and prevent contagion. '
        'Systemic risk is a real and present concern. '
        'We will do whatever it takes to stabilize financial conditions. '
        'Emergency rate action may be considered at the next intermeeting call.'
    ),
    'qa_section': (
        'Q: Are you considering emergency rate cuts? '
        'A: We are not ruling anything out. The financial stability concerns outweigh inflation '
        'considerations at this moment. We are acting urgently and decisively. '
        'Q: Is there a risk of bank runs? '
        'A: We are monitoring systemic risk very carefully and have activated crisis measures.'
    ),
}

print(f"Event: {event_raw['title']}")
print(f"Time: {event_raw['event_time']}")

## 3. Construct Pre-Event Market State

In production, this would come from live Bloomberg feeds. Here we use synthetic data with elevated stress.

In [ ]:
event_time = datetime(2024, 3, 9, 19, 0, tzinfo=timezone.utc)

market_state = MarketStateBuilder.build_synthetic(
    as_of=event_time,
    calendar=pipeline.calendar,
    stress_level=0.65,  # Elevated stress pre-event
    seed=42,
)

print('Pre-event market snapshot:')
print(f'  Session:       {market_state.session_state.value}')
print(f'  VIX:           {market_state.volatility.vix_spot:.1f}')
print(f'  HY Spread:     {market_state.credit.hy_spread_bps:.0f} bps')
print(f'  SPX:           {market_state.equity.spx_level:.0f}')
print(f'  10Y Yield:     {market_state.yields.y10:.2f}%')
print(f'  2s10s Slope:   {market_state.yields.slope_2_10:.0f} bps')
print(f'  DXY:           {market_state.fx.dxy_index:.1f}')
print(f'  Gold:          ${market_state.commodities.gold_spot:.0f}')
print(f'  Data Quality:  {market_state.data_completeness:.0%}')

## 4. Run the Full Pipeline

In [ ]:
ctx = pipeline.process_raw_event(event_raw, market_state=market_state)

print('Pipeline complete.')
print(f'Completed stages: {ctx.completed_stages}')
print(f'Failed stages:    {ctx.failed_stages}')
if ctx.warnings:
    print(f'Warnings:         {ctx.warnings}')

## 5. Event Classification

In [ ]:
e = ctx.event
print(f'Event ID:          {e.event_id[:8]}...')
print(f'Type:              {e.event_type.value}')
print(f'Severity:          {e.severity.value} ({e.severity_score:.1f}/100)')
print(f'Is Weekend:        {e.is_weekend}')
print(f'Full Weekend Gap:  {e.full_weekend_gap}')
print(f'Session State:     {e.market_session_at_event.value}')
print(f'Hours to Open:     {e.hours_until_next_open:.1f}h')

## 6. NLP Analysis

In [ ]:
ps = ctx.policy_surprise
hd = ps.hawkish_dovish

print('=== POLICY SURPRISE VECTOR ===')
print(f'Stance:                {hd.stance.value}')
print(f'HD Score:              {hd.overall_score:+.3f} (−1=dovish, +1=hawkish)')
print(f'Confidence:            {hd.confidence:.0%}')
print(f'Crisis Language:       {hd.crisis_language_detected}')
print(f'Policy Reversal:       {hd.policy_reversal_language}')
print(f'Forward Guidance Chg:  {hd.forward_guidance_change}')
print()
print(f'Composite Surprise:    {ps.composite_surprise_magnitude:.3f}')
print(f'Net Direction:         {ps.net_direction:+.3f}')
print(f'Rate Path Surprise:    {ps.rate_path_surprise:+.3f}')
print(f'Financial Stability:   {ps.financial_stability_surprise:+.3f}')
print(f'Urgency:               {ps.urgency_surprise:.3f}')
print()
print('Key hawkish phrases:', hd.hawkish_phrases[:3])
print('Key dovish phrases: ', hd.dovish_phrases[:3])

## 7. Risk Score Decomposition

In [ ]:
rs = ctx.risk_score

print(f'=== COMPOSITE RISK SCORE: {rs.composite_score:.1f}/100 [{rs.severity.value}] ===')
print(f'Regime:            {rs.regime.value}')
print(f'Regime Multiplier: {rs.regime_multiplier:.2f}×')
print(f'Gap Multiplier:    {rs.gap_risk_multiplier:.2f}×')
print(f'Action Level:      {rs.recommended_action_level}')
print(f'Reliability:       {rs.score_reliability}')
print()

sub_scores = [
    rs.liquidity_risk, rs.volatility_risk, rs.rate_shock_risk,
    rs.equity_downside_risk, rs.credit_spread_risk, rs.fx_risk,
    rs.commodity_shock_risk, rs.weekend_gap_risk, rs.policy_ambiguity_risk,
]

print(f'{"Sub-Score":<30} {"Score":>6}  {"Weight":>7}  {"Contrib":>7}')
print('-' * 58)
for ss in sorted(sub_scores, key=lambda s: s.weighted_contribution, reverse=True):
    print(f'{ss.name:<30} {ss.score:>5.1f}   {ss.weight:>6.2%}  {ss.weighted_contribution:>6.1f}')

In [ ]:
# Waterfall chart of sub-score contributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: sub-score bar chart
names = [ss.name.replace(' Risk', '').replace(' Shock', '') for ss in sub_scores]
scores = [ss.score for ss in sub_scores]
colors = ['#dc2626' if s >= 70 else '#ea580c' if s >= 50 else '#ca8a04' if s >= 35 else '#3b82f6' for s in scores]

sorted_pairs = sorted(zip(scores, names, colors), reverse=True)
scores_s, names_s, colors_s = zip(*sorted_pairs)

bars = ax1.barh(names_s, scores_s, color=colors_s, edgecolor='white', linewidth=0.5)
ax1.axvline(x=75, color='#dc2626', linestyle='--', alpha=0.7, label='CRITICAL threshold')
ax1.axvline(x=55, color='#ea580c', linestyle='--', alpha=0.5, label='HIGH threshold')
ax1.set_xlabel('Sub-Score [0-100]')
ax1.set_title(f'Risk Sub-Scores\nComposite: {rs.composite_score:.1f}/100 [{rs.severity.value}]', fontweight='bold')
ax1.set_xlim(0, 100)
ax1.legend(fontsize=8)
for bar, score in zip(bars, scores_s):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{score:.1f}',
             va='center', fontsize=9)

# Right: scenario probability chart
tree = ctx.scenario_tree
scen_names = [s.name.replace('Significant ', 'Sig. ').replace('Extraordinary ', 'Extr. ') 
              for s in tree.scenarios]
probs = [s.probability for s in tree.scenarios]
eq_impacts = [s.equity_impact_pct for s in tree.scenarios]
scen_colors = ['#dc2626' if s.is_tail_scenario else '#3b82f6' for s in tree.scenarios]

sorted_scen = sorted(zip(probs, scen_names, scen_colors, eq_impacts), reverse=True)
probs_s, snames_s, scolors_s, impacts_s = zip(*sorted_scen)

bars2 = ax2.barh(snames_s, probs_s, color=scolors_s, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Probability')
ax2.set_title(f'Scenario Tree\nExpected Equity: {tree.expected_equity_impact_pct:+.1f}%  |  5th Pct: {tree.tail_loss_5pct:.1f}%', fontweight='bold')
ax2.set_xlim(0, max(probs_s) * 1.3)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.0%}'))

red_patch = mpatches.Patch(color='#dc2626', label='Tail scenario')
blue_patch = mpatches.Patch(color='#3b82f6', label='Base scenario')
ax2.legend(handles=[red_patch, blue_patch], fontsize=8)

for bar, prob, impact in zip(bars2, probs_s, impacts_s):
    ax2.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{prob:.1%}  ({impact:+.1f}%)', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/msre_event_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /tmp/msre_event_analysis.png')

## 8. Portfolio Impact

In [ ]:
pi = ctx.portfolio_impact

print(f'=== PORTFOLIO IMPACT: [{pi.action_level}] ===')
print(f'Alert Level:        {pi.alert_level.value}')
print(f'Kill Switch:        {pi.trigger_kill_switch}')
if pi.recommended_gross_exposure_change:
    print(f'Gross Exposure Δ:   {pi.recommended_gross_exposure_change:.0%}')
if pi.recommended_leverage_change:
    print(f'Leverage Δ:         {pi.recommended_leverage_change:.0%}')
if pi.estimated_drawdown_risk:
    print(f'Drawdown Risk Est.: {pi.estimated_drawdown_risk:.1f}%')
print()
print('--- Equity Guidance ---')
print(pi.equity_guidance[:200] + '...' if len(pi.equity_guidance) > 200 else pi.equity_guidance)
print()
print('--- Rates Guidance ---')
print(pi.rates_guidance[:200] + '...' if len(pi.rates_guidance) > 200 else pi.rates_guidance)

In [ ]:
print(f'=== HEDGE RECOMMENDATIONS ({len(pi.hedge_recommendations)}) ===')
for i, h in enumerate(pi.hedge_recommendations, 1):
    print(f'\n{i}. [{h.urgency}] {h.action} {h.asset_class.value.upper()}')
    print(f'   Instrument: {h.instrument_description}')
    print(f'   Sizing:     {h.sizing_guidance}')
    if h.estimated_cost_bps:
        print(f'   Est. Cost:  {h.estimated_cost_bps:.0f}bps')
    print(f'   Rationale:  {h.rationale[:100]}...')
    print(f'   PM Approval Required: {h.requires_pm_approval}')

## 9. Stress Test: Score vs. Market Stress Level

How does the composite score change as we vary the pre-event market stress level?

In [ ]:
stress_levels = np.linspace(0.05, 0.95, 15)
scores_by_stress = []

for stress in stress_levels:
    ms = MarketStateBuilder.build_synthetic(
        as_of=event_time,
        calendar=pipeline.calendar,
        stress_level=float(stress),
        seed=42,
    )
    c = pipeline.process_raw_event(event_raw, market_state=ms)
    if c.risk_score:
        scores_by_stress.append(c.risk_score.composite_score)
    else:
        scores_by_stress.append(np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stress_levels * 100, scores_by_stress, 'b-o', linewidth=2, markersize=6)
ax.axhline(75, color='#dc2626', linestyle='--', label='CRITICAL threshold (75)')
ax.axhline(55, color='#ea580c', linestyle='--', label='HIGH threshold (55)')
ax.axhline(35, color='#ca8a04', linestyle='--', label='MEDIUM threshold (35)')
ax.fill_between(stress_levels * 100, 75, 100, alpha=0.08, color='#dc2626')
ax.fill_between(stress_levels * 100, 55, 75, alpha=0.08, color='#ea580c')
ax.fill_between(stress_levels * 100, 35, 55, alpha=0.08, color='#ca8a04')
ax.set_xlabel('Pre-Event Market Stress Level (%)', fontsize=12)
ax.set_ylabel('Composite Risk Score [0-100]', fontsize=12)
ax.set_title('Risk Score Sensitivity to Pre-Event Market Stress\n(Weekend Emergency Fed Statement)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 100)
ax.set_xlim(0, 100)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Compare All Scenarios Side-by-Side

In [ ]:
from examples.run_end_to_end import SCENARIOS

scenario_results = {}
for name, raw in SCENARIOS.items():
    ms = MarketStateBuilder.build_synthetic(
        as_of=event_time,
        calendar=pipeline.calendar,
        stress_level=0.40,
        seed=42,
    )
    c = pipeline.process_raw_event(raw, market_state=ms)
    if c.risk_score:
        scenario_results[name] = {
            'score': c.risk_score.composite_score,
            'severity': c.risk_score.severity.value,
            'action': c.risk_score.recommended_action_level,
            'stance': c.policy_surprise.hawkish_dovish.stance.value if c.policy_surprise and c.policy_surprise.hawkish_dovish else 'N/A',
            'gap': c.event.full_weekend_gap if c.event else False,
        }

print(f'{"Scenario":<35} {"Score":>6}  {"Severity":<15} {"Action":<22} {"Stance":<15} {"WkndGap"}')
print('-' * 110)
for name, r in sorted(scenario_results.items(), key=lambda x: x[1]['score'], reverse=True):
    print(f'{name:<35} {r["score"]:>5.1f}   {r["severity"]:<15} {r["action"]:<22} {r["stance"]:<15} {str(r["gap"])}')

## 11. Backtest Summary

In [ ]:
from macro_shock.backtesting.event_study import BacktestEngine, TimestampGuard, TransactionCostModel
from macro_shock.data_schema.models import BacktestEvent, EventType
from synthetic_data_generator import generate_synthetic_events

df = generate_market_data_df(n_days=1000, stress_profile='normal')
guard = TimestampGuard(df, timestamp_col='timestamp')
engine = BacktestEngine(pipeline=pipeline, market_data_guard=guard)

raw_events = generate_synthetic_events()
backtest_events = []
for e in raw_events:
    try:
        be = BacktestEvent(
            event_id=e['event_id'],
            event_date=e['event_date'],
            event_type=EventType(e['event_type']),
            is_weekend=e['is_weekend'],
            institution=e['institution'],
            speaker=e.get('speaker'),
            description=e['description'],
            realized_spx_next_session_return=e.get('realized_spx_next_session_return'),
            realized_10y_yield_change_bps=e.get('realized_10y_yield_change_bps'),
            realized_vix_change=e.get('realized_vix_change'),
            trading_halt_occurred=e.get('trading_halt_occurred', False),
            data_validated=True,
        )
        backtest_events.append(be)
    except Exception as ex:
        print(f'Skip {e["event_id"]}: {ex}')

result = engine.run(backtest_events)
print('=== BACKTEST RESULTS ===')
print(f'Events processed:       {result.n_events}')
print(f'Weekend events:         {result.n_weekend_events}')
print(f'Score accuracy (corr):  {result.score_accuracy:.3f}')
print(f'Precision@CRITICAL:     {result.precision_at_critical:.3f}')
print(f'Recall@CRITICAL:        {result.recall_at_critical:.3f}')
print(f'Expected Shortfall 5%:  {result.expected_shortfall_5pct:.3f}')
print(f'Max Drawdown:           {result.max_drawdown:.3f}')
print(f'Gap Est. Error:         {result.avg_weekend_gap_estimate_error:.2f}%')